<a href="https://www.kaggle.com/code/avtnshm/ssl-geometry-v2-multiseed-baseline?scriptVersionId=299744174" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [1]:
# =========================================================
# Setup
# =========================================================

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as T
from torchvision.models import resnet18
from torch.utils.data import DataLoader
import numpy as np
import random
import pandas as pd
import scipy.stats as stats
import time

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# =========================================================
# Reproducibility
# =========================================================

def set_seed(seed):
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    random.seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

print("Device:", device)
print("Seed function ready.")

Device: cuda
Seed function ready.


In [2]:
# =========================================================
# Data Transforms
# =========================================================

class TwoCropsTransform:
    def __init__(self, base_transform):
        self.base_transform = base_transform

    def __call__(self, x):
        return self.base_transform(x), self.base_transform(x)


ssl_base_transform = T.Compose([
    T.RandomResizedCrop(32, scale=(0.2, 1.0)),
    T.RandomHorizontalFlip(),
    T.ColorJitter(0.4,0.4,0.4,0.1),
    T.RandomGrayscale(p=0.2),
    T.ToTensor()
])

ssl_transform = TwoCropsTransform(ssl_base_transform)

supervised_transform = T.ToTensor()

In [3]:
# =========================================================
# Encoder
# =========================================================

class Encoder(nn.Module):
    def __init__(self):
        super().__init__()
        backbone = resnet18(weights=None)
        backbone.fc = nn.Identity()
        self.backbone = backbone

    def forward(self, x):
        return self.backbone(x)

In [4]:
# =========================================================
# BYOL
# =========================================================

class MLP(nn.Module):
    def __init__(self, in_dim, hidden_dim=2048, out_dim=256):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, out_dim)
        )

    def forward(self, x):
        return self.net(x)


class BYOL(nn.Module):
    def __init__(self):
        super().__init__()
        self.online_encoder = Encoder()
        self.online_projector = MLP(512)
        self.online_predictor = MLP(256, 512, 256)

        self.target_encoder = Encoder()
        self.target_projector = MLP(512)

        for p in self.target_encoder.parameters():
            p.requires_grad = False
        for p in self.target_projector.parameters():
            p.requires_grad = False

    @torch.no_grad()
    def update_target(self, tau=0.996):
        for online, target in zip(self.online_encoder.parameters(),
                                  self.target_encoder.parameters()):
            target.data = tau * target.data + (1 - tau) * online.data

    def forward(self, x1, x2):
        o1 = self.online_predictor(
            self.online_projector(self.online_encoder(x1))
        )

        with torch.no_grad():
            t2 = self.target_projector(
                self.target_encoder(x2)
            )

        loss = 2 - 2 * F.cosine_similarity(o1, t2.detach(), dim=-1)
        return loss.mean()

In [5]:
# =========================================================
# SimCLR
# =========================================================

class SimCLR(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = Encoder()
        self.projector = MLP(512)

    def forward(self, x):
        z = self.projector(self.encoder(x))
        z = F.normalize(z, dim=1)
        return z


def nt_xent_loss(z1, z2, temperature=0.5):
    N = z1.size(0)
    z = torch.cat([z1, z2], dim=0)

    sim = torch.mm(z, z.t()) / temperature
    mask = torch.eye(2*N, device=z.device).bool()
    sim.masked_fill_(mask, -9e15)

    positives = torch.cat([torch.arange(N, 2*N),
                           torch.arange(0, N)]).to(z.device)

    loss = F.cross_entropy(sim, positives)
    return loss

In [6]:
# =========================================================
# Training
# =========================================================

def train_byol(model, train_loader, epochs=100):
    model = model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=3e-4)

    for epoch in range(epochs):
        model.train()
        total_loss = 0

        for (x1, x2), _ in train_loader:
            x1 = x1.to(device)
            x2 = x2.to(device)

            loss = model(x1, x2)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            model.update_target()

            total_loss += loss.item()

        print(f"[BYOL] Epoch {epoch+1:03d}/{epochs} | Loss: {total_loss/len(train_loader):.4f}")

    return model.online_encoder


def train_simclr(model, train_loader, epochs=100):
    model = model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=3e-4)

    for epoch in range(epochs):
        model.train()
        total_loss = 0

        for (x1, x2), _ in train_loader:
            x1 = x1.to(device)
            x2 = x2.to(device)

            z1 = model(x1)
            z2 = model(x2)

            loss = nt_xent_loss(z1, z2)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        print(f"[SimCLR] Epoch {epoch+1:03d}/{epochs} | Loss: {total_loss/len(train_loader):.4f}")

    return model.encoder

In [7]:
# =========================================================
# Linear Probe
# =========================================================

class LinearProbe(nn.Module):
    def __init__(self, encoder):
        super().__init__()
        self.encoder = encoder
        for p in self.encoder.parameters():
            p.requires_grad = False
        self.fc = nn.Linear(512, 10)

    def forward(self, x):
        with torch.no_grad():
            feat = self.encoder(x)
        return self.fc(feat)


def train_probe(encoder, train_loader, test_loader, epochs=10):
    model = LinearProbe(encoder).to(device)
    optimizer = torch.optim.Adam(model.fc.parameters(), lr=1e-3)
    criterion = nn.CrossEntropyLoss()

    for epoch in range(epochs):
        model.train()
        for x, y in train_loader:
            x, y = x.to(device), y.to(device)

            logits = model(x)
            loss = criterion(logits, y)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for x, y in test_loader:
            x, y = x.to(device), y.to(device)
            preds = model(x).argmax(1)
            correct += (preds == y).sum().item()
            total += y.size(0)

    return correct / total

In [8]:
# =========================================================
# Metrics
# =========================================================

tensor_aug = T.Compose([
    T.RandomResizedCrop(32, scale=(0.2, 1.0)),
    T.RandomHorizontalFlip(),
    T.ColorJitter(0.4,0.4,0.4,0.1),
    T.RandomGrayscale(p=0.2),
])

def jacobian_frobenius(model, x):
    x = x.clone().detach().to(device).requires_grad_(True)
    y = model(x)
    v = torch.randn_like(y)

    Jv = torch.autograd.grad(
        outputs=y,
        inputs=x,
        grad_outputs=v,
        retain_graph=False
    )[0]

    return torch.sqrt(Jv.pow(2).sum() / x.size(0))


def compute_metrics(encoder, loader):
    encoder = encoder.to(device)
    encoder.eval()

    jac_vals = []
    aug_vals = []
    noise_vals = []

    for i, (x, _) in enumerate(loader):
        if i == 5:
            break

        x = x.to(device)

        jac_vals.append(jacobian_frobenius(encoder, x).item())

        x_aug = torch.stack([tensor_aug(img.cpu()) for img in x]).to(device)

        with torch.no_grad():
            f1 = encoder(x)
            f2 = encoder(x_aug)
            aug_vals.append(torch.norm(f1 - f2, dim=1).mean().item())

        noise = torch.randn_like(x) * 0.1
        with torch.no_grad():
            f1 = encoder(x)
            f2 = encoder(x + noise)
            noise_vals.append(torch.norm(f1 - f2, dim=1).mean().item())

    return np.mean(jac_vals), np.mean(aug_vals), np.mean(noise_vals)

In [9]:
# =========================================================
# V2 Multi-Seed Experiment (With Timing)
# =========================================================

seeds = [0, 1, 2 ,3]
all_results = []

overall_start = time.time()

for seed in seeds:

    seed_start = time.time()

    print("\n====================================")
    print(f"Running Seed: {seed}")
    print("====================================")

    set_seed(seed)

    train_dataset = torchvision.datasets.CIFAR10(
        root="./data",
        train=True,
        download=True,
        transform=ssl_transform
    )

    supervised_train_dataset = torchvision.datasets.CIFAR10(
        root="./data",
        train=True,
        download=False,
        transform=supervised_transform
    )

    test_dataset = torchvision.datasets.CIFAR10(
        root="./data",
        train=False,
        download=True,
        transform=supervised_transform
    )

    train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True, num_workers=2)
    supervised_train_loader = DataLoader(supervised_train_dataset, batch_size=128, shuffle=True, num_workers=2)
    supervised_test_loader = DataLoader(test_dataset, batch_size=128, shuffle=False, num_workers=2)

    byol_encoder = train_byol(BYOL(), train_loader, epochs=100)
    simclr_encoder = train_simclr(SimCLR(), train_loader, epochs=100)

    byol_acc = train_probe(byol_encoder, supervised_train_loader, supervised_test_loader)
    simclr_acc = train_probe(simclr_encoder, supervised_train_loader, supervised_test_loader)

    byol_jac, byol_aug, byol_noise = compute_metrics(byol_encoder, supervised_test_loader)
    simclr_jac, simclr_aug, simclr_noise = compute_metrics(simclr_encoder, supervised_test_loader)

    seed_time = (time.time() - seed_start) / 60

    all_results.append({
        "Seed": seed,
        "BYOL_Jacobian": byol_jac,
        "SimCLR_Jacobian": simclr_jac,
        "BYOL_Aug": byol_aug,
        "SimCLR_Aug": simclr_aug,
        "BYOL_Noise": byol_noise,
        "SimCLR_Noise": simclr_noise,
        "BYOL_Acc": byol_acc,
        "SimCLR_Acc": simclr_acc,
        "Seed_Time_Min": seed_time
    })

    print(f"\nSeed {seed} finished in {seed_time:.2f} minutes.")

total_time = (time.time() - overall_start) / 60

results_df = pd.DataFrame(all_results)

print("\nPer-Seed Results:")
print(results_df)

print("\nMean Across Seeds:")
print(results_df.mean(numeric_only=True))

print("\nStd Across Seeds:")
print(results_df.std(numeric_only=True))

print(f"\nTotal Experiment Time: {total_time:.2f} minutes.")


Running Seed: 0
[BYOL] Epoch 001/100 | Loss: 0.7509
[BYOL] Epoch 002/100 | Loss: 0.5879
[BYOL] Epoch 003/100 | Loss: 0.5249
[BYOL] Epoch 004/100 | Loss: 0.5003
[BYOL] Epoch 005/100 | Loss: 0.4790
[BYOL] Epoch 006/100 | Loss: 0.4628
[BYOL] Epoch 007/100 | Loss: 0.4508
[BYOL] Epoch 008/100 | Loss: 0.4436
[BYOL] Epoch 009/100 | Loss: 0.4360
[BYOL] Epoch 010/100 | Loss: 0.4333
[BYOL] Epoch 011/100 | Loss: 0.4280
[BYOL] Epoch 012/100 | Loss: 0.4284
[BYOL] Epoch 013/100 | Loss: 0.4263
[BYOL] Epoch 014/100 | Loss: 0.4233
[BYOL] Epoch 015/100 | Loss: 0.4237
[BYOL] Epoch 016/100 | Loss: 0.4208
[BYOL] Epoch 017/100 | Loss: 0.4186
[BYOL] Epoch 018/100 | Loss: 0.4190
[BYOL] Epoch 019/100 | Loss: 0.4166
[BYOL] Epoch 020/100 | Loss: 0.4167
[BYOL] Epoch 021/100 | Loss: 0.4148
[BYOL] Epoch 022/100 | Loss: 0.4157
[BYOL] Epoch 023/100 | Loss: 0.4143
[BYOL] Epoch 024/100 | Loss: 0.4163
[BYOL] Epoch 025/100 | Loss: 0.4159
[BYOL] Epoch 026/100 | Loss: 0.4147
[BYOL] Epoch 027/100 | Loss: 0.4124
[BYOL] Epoc